# NumCompute quickstart

Read one CSV, explore each module, then reuse the same operations in a Pipeline. Finish by comparing NumCompute with Python loops. Sections 1-7 use `demo/data/sample.csv`. Section 8 uses the existing benchmark runner's separately generated, fixed-seed datasets and full-size configuration.

## 0. Set up and run

From the project root, install the package with `python -m pip install -e ".[test]"` and the notebook tools with `python -m pip install ipython ipykernel`. Open this notebook, select that Python environment, restart the kernel and choose **Run All**. The working directory can be the project root or `demo/`.

The CSV contains 3,000 fictional records: `age` and `income` are features, `city` is the true label, and `city_predict` is a rule-based simulated prediction. Missing cells, repeated incomes and extreme incomes are intentional demonstration cases. No ML model training is required.

In [ ]:
import sys
from pathlib import Path
import numpy as np

root = Path.cwd() if (Path.cwd() / "numcompute").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
sample_path = root / "demo" / "data" / "sample.csv"

from numcompute import io, preprocessing as pp, stats, metrics
from numcompute.utils import iter_batches

np.set_printoptions(precision=3, suppress=True)
# print("Sample file:", sample_path)

## 1. Read CSV (`io.py`)

Load numeric features as floats and city labels as text. The default missing tokens include blank cells, `NA`, `NaN`, `NULL` and `N/A`; numeric missing cells become NaN and missing text becomes `''`. Column names let us select features without relying on column positions.

In [ ]:
print("Columns:", io.read_header(sample_path))
X_num, num_names = io.load_csv(
    sample_path, usecols=["age", "income"], dtype=float, return_header=True,
)
city = io.load_csv(sample_path, usecols="city")
city_predictions = io.load_csv(sample_path, usecols="city_predict")

print("Numeric columns and shape:", num_names, X_num.shape)
print("Missing per numeric column:", np.isnan(X_num).sum(axis=0))
print("Blank city labels:", (city == "").sum())
print("First five age/income rows:\n", X_num[:5])

### 1.1 Skip incomplete rows or read in chunks

Skipping removes rows with missing values in the selected columns. Chunking keeps all rows but reads at most 500 input lines at once, which is useful for larger files. Here we stack chunks only to verify that they reproduce the batch result, including NaNs.

In [ ]:
complete_numbers = io.load_csv(
    sample_path, usecols=num_names, dtype=float, on_missing="skip",
)
chunks = list(io.iter_csv_chunks(
    sample_path, chunk_size=500, usecols=num_names, dtype=float,
))
print("Complete numeric rows:", complete_numbers.shape)
print("Chunk shapes:", [chunk.shape for chunk in chunks])
print("Chunks match batch read:", np.array_equal(np.vstack(chunks), X_num, equal_nan=True))

## 2. Preprocess (`preprocessing.py`)

`fit` learns parameters, `transform` applies them, and `fit_transform` does both. First explore the entire CSV; Section 7 separately fits only training rows before processing test rows. Median imputation keeps every row and fills each numeric column from its own observed values.

In [ ]:
imputer = pp.SimpleImputer(strategy="median")
X_full = imputer.fit_transform(X_num)
print("Median fill values:", imputer.statistics_)
print("Missing values after filling:", np.isnan(X_full).sum())

scaler = pp.StandardScaler().fit(X_full)
Z = scaler.transform(X_full)
minmax = pp.MinMaxScaler(feature_range=(0, 1)).fit_transform(X_full)
print("Standardized column means:", Z.mean(axis=0))
print("Standardized column stds:", Z.std(axis=0))

### 2.1 Read the scaling outputs

StandardScaler measures distance from each column's mean in standard-deviation units. MinMaxScaler maps training minima/maxima to 0/1. Both are affected by extreme incomes: the table shows a typical-income row and the highest-income row so the different output scales are visible.

In [ ]:
income = X_full[:, 1]                       # keep original CSV row order
typical_row = np.argmin(np.abs(income - np.median(income)))
highest_row = np.argmax(income)
rows_to_show = [typical_row, highest_row]
print("Columns: original income | StandardScaler | MinMaxScaler")
print(np.column_stack((income, Z[:, 1], minmax[:, 1]))[rows_to_show])

# A scaler fitted on two columns rejects an incompatible feature count.
try:
    scaler.transform(X_full[:, :1])
except ValueError as err:
    print("ValueError:", err)

### 2.2 Encode categorical cities

OneHotEncoder creates one indicator column per learned city. A known city activates one column, while a missing city produces all zeros. With `handle_unknown='ignore'`, an unseen city is also encoded as zeros. The blank example below is selected from the actual CSV; Section 7 will fill it with `Unknown` before encoding.

In [ ]:
encoder = pp.OneHotEncoder(handle_unknown="ignore").fit(city)
blank_rows = np.flatnonzero(city[:, 0] == "")
example_cities = np.vstack((city[:2], city[blank_rows[:1]]))
print("Category order:", encoder.categories_[0])
print("Example city labels:", example_cities.ravel())
print("Encoded examples:\n", encoder.transform(example_cities))

# Hold Sydney out of fit so an unseen category still comes from this CSV.
unseen_encoder = pp.OneHotEncoder(handle_unknown="ignore").fit(city[city[:, 0] != "Sydney"])
print("Sydney when unseen at fit:", unseen_encoder.transform(city[city[:, 0] == "Sydney"][:1]))

## 3. Descriptive statistics (`stats.py`)

For `(rows, features)` input, `axis=0` returns one result per feature. NaNs propagate by default; `skip_nan=True` summarizes only observed values in each column. Mean/median describe location, standard deviation/variance describe spread, and min/max show the range.

In [ ]:
print("Columns:", num_names)
print("Mean (NaNs propagate):", stats.mean(X_num, axis=0))
print("Mean (skip NaNs):", stats.mean(X_num, axis=0, skip_nan=True))
print("Median:", stats.median(X_num, axis=0, skip_nan=True))
print("Sample std:", stats.std(X_num, axis=0, ddof=1, skip_nan=True))
print("Min:", stats.min(X_num, axis=0, skip_nan=True))
print("Max:", stats.max(X_num, axis=0, skip_nan=True))

### 3.1 Quantiles, outliers and histogram

First calculate mean and median using all known incomes, including the extreme values (Group 1). Then exclude incomes above the 99th percentile and calculate the same statistics again (Group 2). Comparing the two groups shows how much the mean and median change when the highest incomes are removed; median can also change, but is usually less sensitive. This comparison trims the top approximately 1% rather than defining a general outlier-detection rule.

Quartiles mark the 25th, 50th and 75th percentiles. The histogram still uses all known incomes so its high-income bins show the original long tail.

In [ ]:
known_income = X_num[:, 1][~np.isnan(X_num[:, 1])]

# Group 1: keep all observed incomes, including the extreme values.
print("Group 1: all known incomes")
print("Rows:", len(known_income))
print("Mean:", stats.mean(known_income))
print("Median:", stats.median(known_income))

# Group 2: exclude the highest approximately 1% of incomes.
cutoff = stats.quantile(known_income, 0.99)
lower_99 = known_income[known_income <= cutoff]
print("\nGroup 2: incomes at or below the 99th percentile")
print("Rows:", len(lower_99), "| cutoff:", cutoff)
print("Mean:", stats.mean(lower_99))
print("Median:", stats.median(lower_99))

print("\nOriginal income quartiles:", stats.quantile(known_income, [0.25, 0.5, 0.75]))
counts, edges = stats.histogram(known_income, bins=6)
print("Original bin edges:", edges)
print("Original counts:", counts, "| total:", counts.sum())

### 3.2 Streaming mean and variance

RunningStats merges batches without retaining previous rows. Its `skip_nan=True` discards an entire row if any feature is missing, unlike the column-wise reductions above. Therefore compare its results with the same complete rows, not with independently filtered columns.

In [ ]:
running = stats.RunningStats(skip_nan=True)
for batch in iter_batches(X_num, 500):
    running.update(batch)

print("Retained rows:", running.n)
print("Running mean:", running.mean)
print("Running sample variance:", running.var(ddof=1))
print("Running sample std:", running.std(ddof=1))
print("Matches complete-row means:", np.allclose(running.mean, complete_numbers.mean(axis=0)))

## 4. Sort, search, top-k and rank (`sort_search.py`, `rank.py`)

Use the median-filled income column so row indices still refer to the original CSV. Sorting orders the whole column; top-k finds the highest incomes; quickselect finds a sorted position without fully sorting; binary search locates a value in an already sorted column.

In [ ]:
from numcompute.sort_search import stable_sort, lexsort, topk, quickselect, binary_search
from numcompute.rank import rank, percentile, percentile_rank

sorted_income = stable_sort(income)
top_values, top_indices = topk(income, 5)
row_order = lexsort((income, X_full[:, 0]))       # age first, income second
print("Lowest incomes:", sorted_income[:5])
print("Top five incomes:", top_values)
print("Their original CSV row indices:", top_indices)
print("First rows sorted by age, then income:\n", X_full[row_order[:5]])
print("Upper middle income:", quickselect(income, len(income) // 2))
print("Find the highest income:", binary_search(sorted_income, top_values[0]))

### 4.1 Ties and percentiles

The CSV deliberately contains 21 Melbourne incomes at 90,000 and 14 Sydney incomes at 115,000. Select a few of those rows to read their ranks within the full income column. Average shares tied positions, min uses the first tied position, dense assigns consecutive ranks to distinct values, and ordinal gives separate ranks in input order. Imputed incomes also create ties elsewhere in this column.

In [ ]:
tie_rows = np.concatenate((
    np.flatnonzero(income == 90_000)[:3],
    np.flatnonzero(income == 115_000)[:3],
))
print("Columns: income | average | min | dense | ordinal")
print(np.column_stack((
    income[tie_rows], rank(income)[tie_rows], rank(income, method="min")[tie_rows],
    rank(income, method="dense")[tie_rows], rank(income, method="ordinal")[tie_rows],
)))
print("Income percentiles (25, 50, 75):", percentile(income, [25, 50, 75]))
print("Percentile position of the highest income:", percentile_rank(income, top_values[0]))

## 5. Metrics (`metrics.py`)

Metrics compare true values with estimates. For classification, use the CSV's `city` and `city_predict`; the latter is generated from age/income profiles, not learned by a model. Exclude blank true city labels so missing data is not counted as a seventh class.

In [ ]:
labelled = city[:, 0] != ""
city_true = city[labelled, 0]
city_pred = city_predictions[labelled, 0]
cities = np.unique(city_true)
print("Labelled rows:", len(city_true), "| excluded:", (~labelled).sum())
print("City order:", cities)
print("Accuracy:", round(metrics.accuracy(city_true, city_pred), 3))
print("Confusion matrix (rows=true, columns=predicted):")
print(metrics.confusion_matrix(city_true, city_pred, labels=cities))
print("Precision per city:", metrics.precision(city_true, city_pred, average=None))
print("Recall per city:", metrics.recall(city_true, city_pred, average=None))
print("F1 per city:", metrics.f1(city_true, city_pred, average=None))

### 5.1 Read classification scores

Accuracy is the overall fraction correct; the confusion-matrix diagonal counts correct predictions. Precision asks how many predictions for a city are right, while recall asks how many true residents of that city were found. F1 combines both. Macro averages cities equally, weighted uses their retained sample counts, and micro pools counts. After hiding labels, city counts are no longer exactly equal, so weighted and macro need not match.

In [ ]:
for average in ("macro", "micro", "weighted"):
    print(f"F1 {average:8s}:", round(metrics.f1(city_true, city_pred, average=average), 3))

binary_true = (city_true == "Sydney").astype(int)
binary_pred = (city_pred == "Sydney").astype(int)
print("Sydney vs rest precision:", round(metrics.precision(binary_true, binary_pred), 3))
print("Sydney vs rest recall:", round(metrics.recall(binary_true, binary_pred), 3))
print("Sydney vs rest F1:", round(metrics.f1(binary_true, binary_pred), 3))

### 5.2 Regression: mean squared error

Use known CSV incomes as regression targets and a constant median-income estimate as a simple baseline. MSE averages squared errors in dollars squared; its square root (RMSE) is in dollars and is easier to interpret. This is a baseline demonstration, not a trained income predictor.

In [ ]:
income_estimate = np.full_like(known_income, np.median(known_income))
income_mse = metrics.mse(known_income, income_estimate)
print("Known income rows:", len(known_income))
print(f"Constant median estimate: {income_estimate[0]:,.0f}")
print(f"MSE: {income_mse:,.0f} dollars squared")
print(f"RMSE: {np.sqrt(income_mse):,.0f} dollars")

### 5.3 Binary ROC and AUC

For Sydney versus the rest, use observed income as an illustrative score: raising the threshold predicts fewer rows as Sydney. This tests income's ranking ability, separately from `city_predict`. ROC shows the false/true positive rates; AUC around 0.5 means chance-level ordering and 1 means perfect separation for these samples. Tied scores enter together, and missing incomes are excluded.

In [ ]:
income_scores = X_num[labelled, 1]
known_score = ~np.isnan(income_scores)
fpr, tpr, thresholds = metrics.roc_curve(
    binary_true[known_score], income_scores[known_score],
)
print("Columns: income threshold | false-positive rate | true-positive rate")
print(np.column_stack((thresholds, fpr, tpr))[:5])
print("AUC:", round(metrics.auc(fpr, tpr), 3))

## 6. Finite-difference gradients and Jacobians (`optim.py`)

Evaluate derivatives at one actual CSV row after standardization. Here `f(x) = sum(x**2)` measures squared distance from the standardized centre: its gradient describes how this score changes with each feature. A positive derivative means increasing that feature increases the score locally; a negative derivative means it decreases. Central differences evaluate both sides of x; forward differences compare x with a positive perturbation. Both should approximate the analytical gradient `2*x`.

In [ ]:
from numcompute.optim import grad, jacobian

x = Z[0].copy()                               # first CSV row, standardized

def f(x):
    return np.sum(x**2)

print("Point [standardized age, standardized income]:", x)
print("Central gradient:", grad(f, x, method="central"))
print("Forward gradient:", grad(f, x, method="forward"))
print("Analytical gradient:", 2 * x)
print("Central matches:", np.allclose(grad(f, x), 2 * x))

### 6.1 Jacobian: several outputs

The vector function below returns two illustrative features: squared distance and an age/income interaction. The Jacobian records each output's sensitivity to both inputs. Rows correspond to outputs and columns to inputs, so two outputs and two inputs give shape `(2, 2)`. This is a numerical-derivative demonstration, not a city prediction.

In [ ]:
def F(x):
    return np.array([np.sum(x**2), x[0] * x[1]])

J = jacobian(F, x)
exact_J = np.array([[2 * x[0], 2 * x[1]], [x[1], x[0]]])

# Differentiate each output with respect to a and b:
# Output 1: a**2 + b**2 -> [2*a, 2*b]
# Output 2: a*b         -> [b, a]

print("Estimated Jacobian:\n", J)
print("Forward Jacobian:\n", jacobian(F, x, method="forward"))
print("Analytical Jacobian:\n", exact_J)
print("Shape:", J.shape, "| matches:", np.allclose(J, exact_J))

## 7. Use the Pipeline (`pipeline.py`)

Now repeat the workflow with named components. Transformers provide `fit(X)` and `transform(X)`; estimators provide `fit(X, y)` and `predict(X)`. Pipeline sends each step's output to the next one. Statistics, sorting and metrics are then called on the outputs; they are not transformer steps themselves.

### 7.1 Fit numerical preprocessing, then reuse it

Fit median imputation and standardization on the first 2,100 CSV rows. Transform the remaining 900 rows using those training parameters. Test data is not fitted again, and its transformed means do not need to be zero.

In [ ]:
from numcompute.pipeline import Pipeline

X_train, X_test = X_num[:2100], X_num[2100:]
numeric_pipe = Pipeline([
    ("impute", pp.SimpleImputer(strategy="median")),
    ("scale", pp.StandardScaler()),
])
X_train_ready = numeric_pipe.fit_transform(X_train)
X_test_ready = numeric_pipe.transform(X_test)
print("Training/test shapes:", X_train_ready.shape, X_test_ready.shape)
print("Training means:", stats.mean(X_train_ready, axis=0))
print("Test means:", stats.mean(X_test_ready, axis=0))
print("First processed test rows:\n", X_test_ready[:3])

### 7.2 Fill and encode cities

Repeat categorical preprocessing as a chain: fill blank labels with `Unknown`, then encode them. The encoder now has seven columns, and originally blank rows activate the Unknown column rather than producing all zeros. This transformed feature array is for demonstrating preprocessing; it does not replace the raw labels used for evaluation.

In [ ]:
city_imputer = pp.SimpleImputer(fill_value="Unknown")
city_encoder = pp.OneHotEncoder()
city_pipe = Pipeline([
    ("impute", city_imputer),
    ("encode", city_encoder),
])
cities_ready = city_pipe.fit_transform(city)
print("Blank cities filled:", len(blank_rows))
print("Filled values:", city_imputer.transform(city)[blank_rows[:3]].ravel())
print("Category order:", city_encoder.categories_[0])
print("Encoded shape:", cities_ready.shape)
print("Originally blank rows:\n", cities_ready[blank_rows[:3]])

### 7.3 Model interface: fit and predict

`Example_model` is a small interface demonstration that initially returns the most common training city. Its messages show that Pipeline calls the model's fit and predict after preprocessing. No actual ML model training is required for this assignment, so after demonstrating predict we replace its output with the CSV's simulated `city_predict` for the same test rows. The displayed accuracy therefore evaluates those precomputed predictions, not a model learned by this Pipeline. Blank true labels are excluded.

In [ ]:
class Example_model:
    'Always predict the most common training city.'

    def fit(self, X, y):
        print("model.fit works!")
        labels, counts = np.unique(y, return_counts=True)
        self.label_ = labels[np.argmax(counts)]
        return self

    def predict(self, X):
        print("model.predict works!")
        return np.full(len(X), self.label_)


y_train, y_test = city[:2100, 0], city[2100:, 0]
valid_train, valid_test = y_train != "", y_test != ""
model_pipe = Pipeline([
    ("impute", pp.SimpleImputer(strategy="median")),
    ("scale", pp.StandardScaler()),
    ("model", Example_model()),
])
model_pipe.fit(X_train[valid_train], y_train[valid_train])
# Demonstrate the model's predict() call through Pipeline.
pipeline_predictions = model_pipe.predict(X_test)

# No actual ML model training is required for this assignment.
# Use the CSV's simulated predictions for the corresponding test rows.
pipeline_predictions = city_predictions[len(X_train):, 0]
print("First simulated predictions:", pipeline_predictions[:5])
print("Simulated test accuracy:", round(metrics.accuracy(
    y_test[valid_test], pipeline_predictions[valid_test],
), 3))

### 7.4 From CSV to evaluation

Repeat the complete classification workflow from the file: load age/income, true cities and simulated predictions; split rows; fit imputation, scaling and the demonstration estimator; then call predict on the test features. As in 7.3, replace the placeholder predictions with the matching `city_predict` rows because no actual ML model is trained. Evaluate only test rows with known true cities. The metrics below describe the CSV's simulation, not the fitted placeholder.

Accuracy measures overall correctness. Macro precision, recall and F1 give each city equal weight; the confusion matrix shows which cities are mixed up. This final workflow omits the separate statistics, ranking and derivative examples.

In [ ]:
# Read everything needed for this workflow directly from the sample file.
demo_X = io.load_csv(sample_path, usecols=["age", "income"], dtype=float)
demo_y = io.load_csv(sample_path, usecols="city").ravel()
demo_saved_predictions = io.load_csv(sample_path, usecols="city_predict").ravel()

split = 900
demo_X_train, demo_X_test = demo_X[:split], demo_X[split:]
demo_y_train, demo_y_test = demo_y[:split], demo_y[split:]
labelled_train = demo_y_train != ""

demo_pipe = Pipeline([
    ("impute", pp.SimpleImputer(strategy="median")),
    ("scale", pp.StandardScaler()),
    ("model", Example_model()),
])
demo_pipe.fit(demo_X_train[labelled_train], demo_y_train[labelled_train])
demo_predictions = demo_pipe.predict(demo_X_test)

# The call above demonstrates Pipeline.predict(), without actual ML training.
# Use precomputed CSV predictions, keeping the same test-row order.
demo_predictions = demo_saved_predictions[split:]

labelled_test = demo_y_test != ""
demo_true = demo_y_test[labelled_test]
demo_pred = demo_predictions[labelled_test]
print("Evaluated test rows:", len(demo_true))
print("Accuracy:", round(metrics.accuracy(demo_true, demo_pred), 3))
print("Precision (macro):", round(metrics.precision(demo_true, demo_pred, average="macro"), 3))
print("Recall (macro):", round(metrics.recall(demo_true, demo_pred, average="macro"), 3))
print("F1 (macro):", round(metrics.f1(demo_true, demo_pred, average="macro"), 3))
print("City order:", np.unique(demo_true))
print("Confusion matrix (rows=true, columns=predicted):")
print(metrics.confusion_matrix(demo_true, demo_pred))

## 8. Benchmark (`benchmarking.py` and `benchmark/`)

Use the existing benchmark implementation directly, independently of sample.csv. `benchmark/run_benchmarks.py` prepares the cases; `benchmark/loops.py` supplies Python-loop baselines; `numcompute.benchmarking.compare` checks matching results and measures runtime. First inspect and try the cases on small inputs, then run the full configuration for the final performance table.

### 8.1 Inspect the cases and try them

`build_cases(QUICK, ...)` returns ten cases, each containing `(operation, loop_func, vec_func, notes)`. Print their names to see the operations and input sizes. Then call the MSE loop and NumCompute functions directly to inspect their results. Finally, try all ten cases through compare using QUICK's two measurements per version. A PASS message means the outputs matched before timing; these small trials are for checking the process, not the final performance report. Temporary CSV files remain available throughout the with block.

In [ ]:
import tempfile
from IPython.display import Markdown, display
from benchmark.run_benchmarks import build_cases, QUICK, SEED, run
from numcompute.benchmarking import compare

with tempfile.TemporaryDirectory() as tmp_dir:
    cases = build_cases(QUICK, np.random.default_rng(SEED), tmp_dir)
    print("Cases (small trial inputs):")
    for index, (name, loop_func, vec_func, notes) in enumerate(cases, start=1):
        print(index, name)

    # The fourth case is MSE: inspect both implementations before timing.
    name, loop_func, vec_func, notes = cases[3]
    loop_result = loop_func()
    numpy_result = vec_func()
    print("\nMSE loop result:", loop_result)
    print("MSE NumCompute result:", numpy_result)
    print("Results match:", np.isclose(loop_result, numpy_result))

    # Try every case with the existing result check and timing helper.
    trial_rows = []
    for name, loop_func, vec_func, notes in cases:
        row = compare(name, loop_func, vec_func, repeat=QUICK["repeat"], rtol=1e-6)
        trial_rows.append(row)
        print("PASS:", name)

print("\nExample trial timing:", trial_rows[3])

### 8.2 Complete performance table

Now call `run(quick=False)` to repeat all ten comparisons using the runner's FULL datasets, seed 42 and seven measurements per version. The runner prepares fresh inputs, checks each pair of results, times both implementations and builds the report using the existing benchmarking helpers. This includes the numeric CSV reader, accuracy, confusion matrix, MSE, ROC/AUC, StandardScaler, histogram, stable sort, top-k and average rank.

NumCompute result is the median milliseconds per call; Baseline/result gives the Python-loop time and speedup (`loop_time / NumCompute_time`). A speedup above 1 means NumCompute was faster for that input; below 1 means the loop was faster. Results depend on the reported machine and software environment. Rank, top-k and AUC also compare different algorithms, so their gains are not purely vectorization.

The report below is the full performance table, not the small trial results from 8.1. To write a report to `benchmark/results.md`, run `python -m benchmark.run_benchmarks` from the project root.

In [ ]:
# Use the existing full runner and display all ten measured results.
benchmark_report = run(quick=False)
display(Markdown(benchmark_report))